# AI Interview Question & Rubric Generator

**Goal:** build a tool that takes a **role, seniority, skills, and an optional job description**, and returns **interview questions with scoring rubrics**. 


**Concepts covered**
1. One interface for multiple LLM providers **(OpenAI, Groq, Gemini)** - User can choose anyone
2. Why free-text LLM output is a poor fit for applications
3. Defining output structure with Pydantic
4. Prompt templates
5. Chaining with LangChain Expression Language (LCEL)
6. Structured output: LLM response parsed into a Python object
7. Error handling and retries
8. Mapping this notebook to `app.py`

**Pipeline**

```
Inputs (role, seniority, skills, JD)
        |
        v
Prompt template  ->  LLM (OpenAI | Groq | Gemini)  ->  Pydantic object (InterviewKit)
                                                              |
                                                              v
                                                  Render / export (UI, Markdown, JSON)
```



## 0. Setup

In [2]:
# %pip install langchain==1.3.6 langchain-openai==1.2.2 langchain-groq==1.1.3 langchain-google-genai pydantic pypdf==6.10.2

In [3]:
# %pip install langchain-core==1.6.6

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# check package varsion
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-openai", "langchain-google-genai",
            "langchain-core", "langchain-groq",
            "pypdf","pydantic"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


# langchain version: 1.3.6
# langchain-openai version: 1.2.2
# langchain-google-genai version: 4.4.0
# langchain-core version: 1.6.6
# langchain-groq version: 1.1.3
# pypdf version: 6.10.2
# pydantic version: 2.13.5

langchain version: 1.3.6
langchain-openai version: 1.2.2
langchain-google-genai version: 4.4.0
langchain-core version: 1.6.6
langchain-groq version: 1.1.3
pypdf version: 6.10.2
pydantic version: 2.13.5


In [5]:
import os
import time
from getpass import getpass
from typing import Literal

from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError
from pypdf import PdfReader

## 1. Provider abstraction

All LangChain chat models expose the same methods (`.invoke()`, `.with_structured_output()`). Switching providers means switching the constructor; the rest of the code stays unchanged.

Imports are done **inside** `build_llm` (lazy import), so only the selected provider's package has to be installed and working.

In [6]:
PROVIDERS = {
    "OpenAI": {"model": "gpt-4o-mini", "env": "OPENAI_API_KEY"},
    "Groq":   {"model": "llama-3.3-70b-versatile", "env": "GROQ_API_KEY"},
    "Gemini": {"model": "gemini-2.5-flash", "env": "GOOGLE_API_KEY"},
}


def build_llm(provider: str, model: str, api_key: str, temperature: float):
    if provider == "OpenAI":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, api_key=api_key, temperature=temperature)
        
    if provider == "Groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=model, api_key=api_key, temperature=temperature)
        
    if provider == "Gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, google_api_key=api_key, temperature=temperature)
        
    raise ValueError(f"Unknown provider: {provider}")

### Choose a provider

In the app this is the sidebar selectbox. Here it is a variable. The key is read from an environment variable, or prompted if missing. Never hard-code keys in a notebook.

### `temperature` controls randomness: 
- 0 is near-deterministic,
- 1 is more varied.
- 0.4 balances consistency and variety for question generation.

In [7]:
PROVIDER = "OpenAI"   # User can choose "OpenAI" | "Groq" | "Gemini"

cfg = PROVIDERS[PROVIDER]
api_key = os.getenv(cfg["env"]) or getpass(f"{cfg['env']}: ")
llm = build_llm(PROVIDER, cfg["model"], api_key, temperature=0.4)

OPENAI_API_KEY:  ········


## 2. The **naive** approach: free-text output

Ask the model directly and look at what comes back.

In [5]:
resp = llm.invoke("Generate 2 interview questions with scoring rubrics for a Senior Data Engineer.")
print(resp.content)

Certainly! Here are two interview questions for a Senior Data Engineer position, along with scoring rubrics for each question.

### Question 1: Data Pipeline Design
**Question:** "Can you describe a data pipeline you have designed and implemented? What were the key challenges you faced, and how did you overcome them?"

**Scoring Rubric:**
- **0-1 Points:** The candidate provides a vague or unclear description of a data pipeline, lacks details about challenges, and does not explain how they were addressed.
- **2-3 Points:** The candidate describes a specific data pipeline but provides limited details on the architecture or technologies used. Challenges mentioned are generic, and solutions are not well-explained.
- **4-5 Points:** The candidate provides a clear and detailed description of a specific data pipeline, including the architecture, technologies, and tools used. They articulate specific challenges faced (e.g., data quality, performance issues) and provide thoughtful, well-reason

## **Problems for an application**
- Format changes between runs (numbered list, table, headings, bullets).
- No guaranteed fields, so the UI cannot reliably pull "rubric" or "difficulty".
- Parsing with regex is brittle.
- Nothing validates the content (for example, a rubric with 3 levels instead of 4).

## **Fix** : Define the structure as a schema and make the model return data that matches it.

## 3. Define the output schema (Pydantic)

Pydantic classes describe the shape of the data. LangChain converts them to a JSON schema and sends it to the model.

- `Field(description=...)` is sent to the model, so descriptions work as extra instructions.
- `Literal[...]` restricts a field to fixed values.
- `list[RubricLevel]` nests one model inside another.

In [8]:
class RubricLevel(BaseModel):
    score:      int = Field(description="1 (poor) to 4 (excellent)")
    label:      str = Field(description="Short label, e.g. 'Insufficient'")
    descriptor: str = Field(description="Observable behaviour at this level")


class Question(BaseModel):
    question:            str
    category:            Literal["technical", "behavioral", "situational", "system_design"]
    difficulty:          Literal["easy", "medium", "hard"]
    competency:          str = Field(description="Skill or competency being assessed")
    good_answer_signals: list[str] = Field(description="2-4 signals of a strong answer")
    red_flags:           list[str] = Field(description="1-3 warning signs")
    rubric:              list[RubricLevel] = Field(description="Exactly 4 levels, scores 1-4")


class InterviewKit(BaseModel):
    role:      str
    questions: list[Question]

### Schema validation (no API call needed)

Pydantic rejects data that does not match the schema. This is what protects the app from bad model output.

In [9]:
ok = Question(
    question="How would you deduplicate 1B events in Spark?",
    category="technical",
    difficulty="hard",
    competency="Spark optimization",
    good_answer_signals=["Mentions window functions or dropDuplicates with watermark"],
    red_flags=["Collects data to the driver"],
    rubric=[RubricLevel(score=s, label=l, descriptor="...") 
            for s, l in enumerate(["Poor", "Fair", "Good", "Excellent"], 1)
           ],
)

print("valid:", ok.category)

try:
    Question(question="?",
             category="trivia", # This does NOT exist
             difficulty="easy",
             competency="x",
             good_answer_signals=[],
             red_flags=[],
             rubric=[]
    )
except ValidationError as e:
    print("invalid:", e.errors()[0]["msg"])

valid: technical
invalid: Input should be 'technical', 'behavioral', 'situational' or 'system_design'


## 4. Prompt template

A template separates fixed instructions from variable inputs.

- **system** message: the model's role and rules (stable).
- **human** message: the request with `{placeholders}` filled at runtime.

In [10]:
system_prompt = """
     You are an expert technical recruiter and interview designer.
     Generate role-specific interview questions with scoring rubrics.
     Questions must be concrete, non-generic, and calibrated to the seniority level.
     Each rubric has exactly 4 levels (scores 1-4) with observable, distinct descriptors.
"""

human_prompt = """
     Role: {role}
     Seniority: {seniority}
     Key skills: {skills
     Question categories: {categories}
     Number of questions: {n}
     Job description (optional):{jd}
     Generate the interview kit.
"""



PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt),
])

Inspect the filled prompt before it reaches the model (no API call). This is the first thing to check when output quality is poor.

In [11]:
inputs = dict(
    role="Senior Data Engineer",
    seniority="Senior",
    skills="Python, Spark, Airflow, SQL",
    categories="technical, behavioral",
    n=4,
    jd="not provided", # This can come from a PDF.
)

for m in PROMPT.invoke(inputs).to_messages():
    print(f"[{m.type}]\n{m.content}\n")

[system]

     You are an expert technical recruiter and interview designer.
     Generate role-specific interview questions with scoring rubrics.
     Questions must be concrete, non-generic, and calibrated to the seniority level.
     Each rubric has exactly 4 levels (scores 1-4) with observable, distinct descriptors.


[human]

     Role: Senior Data Engineer

     Seniority: Senior

     Key skills: Python, Spark, Airflow, SQL

     Question categories: technical, behavioral

     Number of questions: 4

     Job description (optional):
not provided


     Generate the interview kit.




## 5. Chain: prompt -> structured LLM

`llm.with_structured_output(InterviewKit)` does three things:
1. Converts the Pydantic class to a JSON schema.
2. Sends it using the provider's native mechanism (tool calling or JSON mode).
3. Parses the reply back into an `InterviewKit` object.

The `|` operator (LCEL) pipes the output of the left component into the right one.

In [12]:
structured_llm = llm.with_structured_output(InterviewKit)
chain = PROMPT | structured_llm


def generate(llm, **inputs) -> InterviewKit:
    chain = PROMPT | llm.with_structured_output(InterviewKit)
    return chain.invoke(inputs)

In [13]:
kit = generate(llm, **inputs)

print(type(kit))
print(len(kit.questions), "questions")
print(kit.questions[0].question)

<class '__main__.InterviewKit'>
4 questions
Describe your experience with Apache Spark. Can you explain how you would optimize a Spark job for performance?


NOTE: `kit` is a Python object, not text. Fields are accessed with dot notation (`kit.questions[0].rubric[0].descriptor`), which is what makes UI rendering straightforward.

In [12]:
print(kit.questions[1].question)

How do you manage data pipelines using Apache Airflow? Can you walk us through a complex DAG you have implemented?


In [13]:
print(kit.questions[2].question)

Can you discuss a challenging data-related problem you faced and how you approached solving it?


In [14]:
print(kit.questions[3].question)

How do you ensure data quality and integrity in your data engineering processes?


## 6. Render and export
- Following generates 4 inerview questions along with its rubric

In [14]:
for i, q in enumerate(kit.questions, 1):
    print(f"Q{i}. {q.question}")
    print(f"    {q.category} | {q.difficulty} | {q.competency}")
    for r in sorted(q.rubric, key=lambda r: r.score):
        print(f"    {r.score} - {r.label}: {r.descriptor}")
    print()

Q1. Describe your experience with Apache Spark. Can you explain how you would optimize a Spark job for performance?
    technical | hard | Spark Optimization
    1 - Insufficient: Unable to explain Spark or optimizations.
    2 - Basic: Mentions some Spark features but lacks depth in optimization techniques.
    3 - Proficient: Describes several optimization techniques with examples but lacks detailed technical depth.
    4 - Excellent: Thoroughly explains multiple advanced optimization techniques, backed by relevant examples.

Q2. How do you handle data pipeline failures in Airflow? Can you walk us through your approach to monitoring and alerting?
    technical | medium | Airflow Management
    1 - Insufficient: No understanding of Airflow failure handling.
    2 - Basic: Mentions some monitoring practices but lacks detail on handling failures.
    3 - Proficient: Describes a structured approach to monitoring and alerting but lacks specific tools.
    4 - Excellent: Thoroughly explain

The same object converts to Markdown or JSON. In the app these feed the download buttons.

In [15]:
def to_markdown(kit: InterviewKit) -> str:
    out = [f"# Interview Kit: {kit.role}\n"]
    for i, q in enumerate(kit.questions, 1):
        out.append(f"## Q{i}. {q.question}")
        out.append(f"*{q.category} | {q.difficulty} | {q.competency}*\n")
        out.append("**Good answer signals**")
        out += [f"- {s}" for s in q.good_answer_signals]
        out.append("\n**Red flags**")
        out += [f"- {s}" for s in q.red_flags]
        out.append("\n**Rubric**\n\n| Score | Label | Descriptor |\n|---|---|---|")
        out += [f"| {r.score} | {r.label} | {r.descriptor} |" for r in sorted(q.rubric, key=lambda r: r.score)]
        out.append("")
    return "\n".join(out)


display(Markdown(to_markdown(kit)))

# Interview Kit: Senior Data Engineer

## Q1. Describe your experience with Apache Spark. Can you explain how you would optimize a Spark job for performance?
*technical | hard | Spark Optimization*

**Good answer signals**
- Mentions specific Spark configurations (e.g., memory settings, partitioning)
- Discusses the use of DataFrames and Catalyst optimizer
- Provides examples of real-world scenarios where optimizations were applied

**Red flags**
- Vague about Spark features
- Cannot provide specific examples of optimizations
- Shows lack of understanding of Spark architecture

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | Unable to explain Spark or optimizations. |
| 2 | Basic | Mentions some Spark features but lacks depth in optimization techniques. |
| 3 | Proficient | Describes several optimization techniques with examples but lacks detailed technical depth. |
| 4 | Excellent | Thoroughly explains multiple advanced optimization techniques, backed by relevant examples. |

## Q2. How do you handle data pipeline failures in Airflow? Can you walk us through your approach to monitoring and alerting?
*technical | medium | Airflow Management*

**Good answer signals**
- Describes specific monitoring tools and practices used
- Mentions strategies for alerting and recovery
- Provides examples of past experiences with pipeline failures

**Red flags**
- Unfamiliar with Airflow's error handling features
- Lacks a structured approach to monitoring
- Does not mention any real-world scenarios

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | No understanding of Airflow failure handling. |
| 2 | Basic | Mentions some monitoring practices but lacks detail on handling failures. |
| 3 | Proficient | Describes a structured approach to monitoring and alerting but lacks specific tools. |
| 4 | Excellent | Thoroughly explains a robust monitoring and alerting strategy with specific tools and past experiences. |

## Q3. Can you explain a complex SQL query you have written? What challenges did you face, and how did you overcome them?
*technical | medium | SQL Querying*

**Good answer signals**
- Describes the purpose of the query and its complexity
- Mentions specific SQL functions or techniques used
- Discusses challenges and how they were resolved

**Red flags**
- Struggles to explain query complexity
- Cannot articulate challenges faced
- Lacks familiarity with advanced SQL functions

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | Cannot explain a complex SQL query. |
| 2 | Basic | Describes a simple query without significant challenges. |
| 3 | Proficient | Explains a complex query with some challenges but lacks depth in resolution. |
| 4 | Excellent | Thoroughly explains a complex SQL query, detailing challenges and effective resolutions. |

## Q4. Describe a time when you had to work collaboratively with a team to solve a data engineering problem. What role did you play, and what was the outcome?
*behavioral | medium | Team Collaboration*

**Good answer signals**
- Clearly defines their role in the collaboration
- Describes the problem and the collaborative process
- Mentions the positive outcome and lessons learned

**Red flags**
- Vague about their role in teamwork
- Cannot articulate the problem or the collaborative process
- No clear outcome or lessons learned

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | No relevant experience in team collaboration. |
| 2 | Basic | Mentions a team experience but lacks detail on their role or the problem. |
| 3 | Proficient | Describes a specific collaboration experience with some detail about their role and outcome. |
| 4 | Excellent | Thoroughly explains a significant collaboration experience, detailing their active role, the problem-solving process, and a positive outcome. |


In [18]:
print(kit.model_dump_json(indent=2)[:800])

{
  "role": "Senior Data Engineer",
  "questions": [
    {
      "question": "Describe your experience with Apache Spark. Can you explain how you would optimize a Spark job for performance?",
      "category": "technical",
      "difficulty": "hard",
      "competency": "Spark optimization",
      "good_answer_signals": [
        "Provides specific examples of Spark jobs they've optimized",
        "Mentions techniques like partitioning, caching, and avoiding shuffles",
        "Demonstrates understanding of Spark's execution model"
      ],
      "red_flags": [
        "Vague or generic answers without specific examples",
        "Lacks understanding of Spark's architecture",
        "Does not mention any optimization techniques"
      ],
      "rubric": [
        {
          "score": 1,



## 7. Adding context from a job description PDF

`pypdf` extracts text, which is injected into the `{jd}` placeholder. The text is truncated (8,000 characters here) to control cost and stay inside the model's context window.

In [16]:
def read_pdf(file) -> str:
    return "\n".join(page.extract_text() or "" for page in PdfReader(file).pages)


JD_PATH = "job_description_senior_data_engineer.pdf"   # e.g. "job_description.pdf"

if JD_PATH:
    jd_text = read_pdf(JD_PATH)[:8000]
    kit_jd = generate(llm, **{**inputs, "jd": jd_text})
    display(Markdown(to_markdown(kit_jd)))
else:
    print("Set JD_PATH to a PDF to try this step.")

# Interview Kit: Senior Data Engineer

## Q1. Describe a challenging data pipeline you designed and implemented. What were the key considerations you took into account, and what technologies did you use?
*technical | hard | Data Pipeline Design*

**Good answer signals**
- Provides a clear overview of the pipeline architecture
- Mentions specific technologies and tools used
- Discusses challenges faced and how they were overcome
- Highlights performance metrics or improvements achieved

**Red flags**
- Vague description of the pipeline
- Lack of specific technologies mentioned
- No clear understanding of challenges and solutions

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | Fails to describe a relevant pipeline or lacks understanding of key concepts. |
| 2 | Basic | Describes a pipeline but lacks depth in technology or challenges faced. |
| 3 | Proficient | Clearly explains a relevant pipeline with specific technologies and challenges addressed. |
| 4 | Excellent | Provides a comprehensive overview of a complex pipeline, detailing technologies, challenges, and measurable outcomes. |

## Q2. How do you approach optimizing Spark jobs for large datasets? Can you provide an example of a specific optimization you implemented?
*technical | hard | Spark Optimization*

**Good answer signals**
- Mentions specific optimization techniques (e.g., partitioning, caching)
- Provides a concrete example of an optimization and its impact
- Demonstrates understanding of Spark internals and performance tuning
- Discusses trade-offs and considerations in optimization

**Red flags**
- No specific examples provided
- Lacks understanding of Spark optimization techniques
- Fails to discuss performance impacts

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | Shows minimal understanding of Spark optimization or provides no examples. |
| 2 | Basic | Describes general optimization concepts but lacks specific examples or depth. |
| 3 | Proficient | Provides a relevant example of optimization with clear techniques and outcomes. |
| 4 | Excellent | Demonstrates deep understanding of Spark optimization with multiple examples and significant performance improvements. |

## Q3. Can you explain how you would implement data quality checks in a data pipeline? What tools or frameworks would you use?
*technical | medium | Data Quality Assurance*

**Good answer signals**
- Mentions specific data quality frameworks or tools (e.g., Great Expectations, dbt)
- Describes a systematic approach to implementing checks
- Discusses how to monitor and alert on data quality issues
- Provides examples of types of checks (e.g., schema validation, anomaly detection)

**Red flags**
- Vague on tools or frameworks used
- Lacks a systematic approach to data quality
- No examples of checks provided

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | Fails to provide a coherent approach to data quality checks. |
| 2 | Basic | Mentions general concepts of data quality but lacks specific tools or examples. |
| 3 | Proficient | Describes a clear approach to data quality with relevant tools and examples. |
| 4 | Excellent | Provides a comprehensive strategy for data quality checks, detailing tools, monitoring, and specific examples. |

## Q4. Describe a situation where you had to mentor a junior engineer. What approach did you take, and what was the outcome?
*behavioral | medium | Mentorship and Leadership*

**Good answer signals**
- Describes a specific mentoring experience
- Highlights the approach taken and resources used
- Discusses the outcome and growth of the mentee
- Reflects on lessons learned from the mentoring experience

**Red flags**
- No specific mentoring experience mentioned
- Lacks a clear approach or methodology
- Fails to discuss outcomes or reflections

**Rubric**

| Score | Label | Descriptor |
|---|---|---|
| 1 | Insufficient | No relevant mentoring experience or understanding of mentorship. |
| 2 | Basic | Mentions mentoring but lacks detail on approach or outcomes. |
| 3 | Proficient | Describes a specific mentoring experience with clear approach and positive outcomes. |
| 4 | Excellent | Provides a detailed account of mentoring, showcasing effective strategies and significant growth in the mentee. |


# STOP

## 8. (OPTIONAL) Same chain, different LLM providers

Because every provider shares the same interface, comparing them is a loop. Providers without a key in the environment are skipped.

In [17]:
for name, c in PROVIDERS.items():
    key = os.getenv(c["env"])
    if not key:
        print(f"{name}: skipped (no {c['env']})")
        continue
    t0 = time.time()
    try:
        k = generate(build_llm(name, c["model"], key, 0.4), **inputs)
        print(f"{name}: {time.time() - t0:.1f}s | {len(k.questions)} questions | Q1: {k.questions[0].question}")
    except Exception as e:
        print(f"{name}: failed ({type(e).__name__})")

OpenAI: skipped (no OPENAI_API_KEY)
Groq: skipped (no GROQ_API_KEY)
Gemini: skipped (no GOOGLE_API_KEY)


## 9. Failure modes and retries

Structured output can fail, mostly with smaller models: invalid JSON, a missing field, or a value outside a `Literal`. 

**Try these options**, in order of effort:

1. Lower the temperature.
2. Use a larger model.
3. Simplify the schema (fewer nested fields).
4. Retry automatically.

In [21]:
resilient_chain = (PROMPT | llm.with_structured_output(InterviewKit)).with_retry(stop_after_attempt=3)
kit = resilient_chain.invoke(inputs)
print(len(kit.questions), "questions")

4 questions


## 10. From notebook to `app.py`

| Notebook step | In `app.py` |
|---|---|
| 1. Providers | `PROVIDERS`, `build_llm`, sidebar selectbox / model / API key |
| 3. Schema | `RubricLevel`, `Question`, `InterviewKit` |
| 4. Prompt | `PROMPT` |
| 5. Chain | `generate()` |
| 6. Render / export | `to_markdown`, expanders, `st.table`, download buttons |
| 7. PDF | `read_pdf`, `st.file_uploader` |
| 9. Errors | `try/except` around `generate`, shown with `st.error` |

**New in the app: Streamlit concepts**
- The script reruns from top to bottom on every user interaction.
- Widgets (`st.text_input`, `st.selectbox`, `st.slider`) return their current value on each run.
- `st.session_state` keeps data between reruns. The generated `kit` is stored there; otherwise it would disappear on the next click (for example, a download button).
- `key=` on widgets keeps separate values per provider (model name and API key per provider).
- `st.sidebar` holds configuration; the main area holds inputs and results.

Run it with `python -m streamlit run app.py`.

## Exercises

1. Add `"coding_challenge"` to `category` and to the app's multiselect.
2. Change the rubric from 4 to 5 levels. Which lines must change (schema description, prompt, table sorting)?
3. Add `estimated_minutes: int` to `Question` and show it in the output.
4. Print the JSON schema LangChain sends to the model: `InterviewKit.model_json_schema()`. Find where your `Field(description=...)` text appears.
5. Set temperature to 0 and 1, generate 3 times each, and compare variation.
6. Stretch: store past questions in Pinecone and retrieve similar ones to avoid duplicates (RAG).